# Homework 2, Part 1: Data Cleaning and Validation

This notebook starts from the raw `ohlc_hw` table in the local `data.db`. It records data-quality findings before repairs, applies documented ticker-local repairs, and validates the result before writing `ohlc_hw_cleaned` and audit tables back to the same database. The source table is read only.

Run the notebook from this homework folder. The database and its archive are local course data and are excluded from Git by `.gitignore`.

## 1.1 Load and inspect the data

The supplied `ohlc` table is used only as a ticker-specific session calendar and, when available, as a same-date source for adjusted-close factors.

### Source columns

| Column | Meaning |
| --- | --- |
| `ts` | Observation date for the trading session. |
| `ticker` | Vendor stock label; normalized before grouping. |
| `open` | Raw opening price. |
| `high` | Raw highest price during the session. |
| `low` | Raw lowest price during the session. |
| `close` | Raw closing price. |
| `adj_close` | Vendor adjusted closing price, used for return reports. |
| `volume` | Reported shares traded; a filled zero remains flagged as unknown volume. |

The cleaned table also preserves `original_ticker` and per-field `*_repaired` flags.

In [1]:
from pathlib import Path
import multiprocessing as mp
import os
import re
import shutil
import sqlite3
import sys
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from concurrent.futures.process import BrokenProcessPool

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

HOMEWORK_DIR = Path.cwd()
if not (HOMEWORK_DIR / "data.db").is_file() and not (HOMEWORK_DIR / "data" / "data.db").is_file():
    candidate = HOMEWORK_DIR / "Homeworks" / "HW2" / "David_MartinsHuber"
    if (candidate / "data.db").is_file() or (candidate / "data" / "data.db").is_file():
        HOMEWORK_DIR = candidate
SOURCE_DB_PATH = HOMEWORK_DIR / "data.db"
WORKING_DATA_DIR = HOMEWORK_DIR / "data"
WORKING_DATA_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = WORKING_DATA_DIR / "data.db"
if not DB_PATH.is_file():
    if not SOURCE_DB_PATH.is_file():
        raise FileNotFoundError(f"Place the course database at {SOURCE_DB_PATH} or {DB_PATH}.")
    shutil.copy2(SOURCE_DB_PATH, DB_PATH)
    print(f"Created working copy at {DB_PATH}")
if str(HOMEWORK_DIR) not in sys.path:
    sys.path.insert(0, str(HOMEWORK_DIR))
from hw2_cleaning_workers import repair_ticker_group

RAW_PRICE_COLUMNS = ["open", "high", "low", "close"]
PRICE_COLUMNS = RAW_PRICE_COLUMNS + ["adj_close"]
REQUIRED_COLUMNS = ["ts", "ticker", *PRICE_COLUMNS, "volume"]
REPAIR_FLAG_COLUMNS = [f"{column}_repaired" for column in PRICE_COLUMNS + ["volume"]]

with sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True) as connection:
    tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", connection)["name"].tolist()
    if "ohlc_hw" not in tables:
        raise ValueError("The local database does not contain the required ohlc_hw table.")
    raw_source = pd.read_sql_query("SELECT * FROM ohlc_hw", connection)
    reference_source = pd.read_sql_query("SELECT * FROM ohlc", connection) if "ohlc" in tables else pd.DataFrame()

print(f"Database: {DB_PATH}")
print(f"Raw rows loaded: {len(raw_source):,}")
print(f"Reference rows loaded: {len(reference_source):,}")
display(raw_source.head())

Database: /Users/davidhuber/Desktop/berkeley prequesites/Preprogram/UCB-MFE-python-preprogram-2026-winter/Homeworks/HW2/David_MartinsHuber/data/data.db
Raw rows loaded: 397,711
Reference rows loaded: 368,251
           ts      open      high  ...  adj_close    volume  ticker
0  1975-04-14    5.4300    5.4800  ...   1.640000    446392     CAT
1  1978-04-05    4.8400    4.8900  ...   0.656990    355200     CVX
2  2018-05-25  124.5400  124.7600  ...  92.220000   8936649     CVX
3  1995-12-07    9.9700    9.9700  ...   5.880000   3954150      HD
4  1981-07-23    0.1038    0.1038  ...   0.080013  34451232    AAPL

[5 rows x 8 columns]


In [2]:
def normalize_ohlc(frame: pd.DataFrame) -> pd.DataFrame:
    # Standardize field names, dates, tickers, and numeric values for analysis.
    data = frame.copy()
    data.columns = [str(column).strip().lower() for column in data.columns]
    aliases = {"date": "ts", "timestamp": "ts", "adjusted_close": "adj_close", "adjclose": "adj_close"}
    data = data.rename(columns={key: value for key, value in aliases.items() if key in data and value not in data})
    missing_columns = set(REQUIRED_COLUMNS).difference(data.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns: {sorted(missing_columns)}")
    data["original_ticker"] = data["ticker"].astype("string")
    data["ts"] = pd.to_datetime(data["ts"], errors="coerce").dt.normalize()
    data["ticker"] = (
        data["ticker"].astype("string")
        .str.replace(r"<[^>]*>", "", regex=True)
        .str.strip().str.upper()
    )
    for column in PRICE_COLUMNS + ["volume"]:
        data[column] = pd.to_numeric(data[column], errors="coerce")
    return data

raw = normalize_ohlc(raw_source)
reference = normalize_ohlc(reference_source) if not reference_source.empty else pd.DataFrame(columns=[*REQUIRED_COLUMNS, "original_ticker"])

print("Raw table schema and non-null counts:")
raw.info()
print(); print("Descriptive statistics:")
display(raw[PRICE_COLUMNS + ["volume"]].describe().T)
print(f"Date range: {raw['ts'].min().date()} to {raw['ts'].max().date()}")
print(f"Normalized tickers ({raw['ticker'].nunique()}): {sorted(raw['ticker'].dropna().unique())}")
print(f"Distinct original ticker labels: {raw['original_ticker'].nunique()}")
display(raw[REQUIRED_COLUMNS].isna().sum().rename("missing_cells").to_frame())

Raw table schema and non-null counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 9 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   ts               397711 non-null  datetime64[ns]
 1   open             397711 non-null  float64       
 2   high             397711 non-null  float64       
 3   low              397711 non-null  float64       
 4   close            397711 non-null  float64       
 5   adj_close        377856 non-null  float64       
 6   volume           397711 non-null  int64         
 7   ticker           397711 non-null  string        
 8   original_ticker  397711 non-null  string        
dtypes: datetime64[ns](1), float64(5), int64(1), string(2)
memory usage: 27.3 MB

Descriptive statistics:
              count          mean  ...          75%           max
open       397711.0  4.628548e+01  ...        57.53  6.042600e+02
high       397711.0 

## 1.2 Diagnose data quality before repairs

Severity is a review guide: high means a price or key cannot safely be used as supplied, medium means the observation needs investigation, and low means a caution such as zero volume. Multiple issue rows may refer to one observation. Missing trading dates are checked against the clean table's dates for the same ticker within its observed homework-data history; this avoids treating exchange holidays as gaps.

In [3]:
ISSUE_COLUMNS = ["ticker", "issue_type", "issue_date", "description", "severity"]


def issue_rows(frame: pd.DataFrame, mask: pd.Series, issue_type: str, severity: str, description) -> pd.DataFrame:
    # Create one standardized quality-report row for every true mask value.
    selected = frame.loc[mask, ["ticker", "ts"]].copy()
    if selected.empty:
        return pd.DataFrame(columns=ISSUE_COLUMNS)
    selected["ticker"] = selected["ticker"].fillna("UNKNOWN").astype(str)
    selected["issue_type"] = issue_type
    selected["issue_date"] = selected["ts"].dt.strftime("%Y-%m-%d").fillna("UNKNOWN")
    selected["description"] = description(selected) if callable(description) else description
    selected["severity"] = severity
    return selected[ISSUE_COLUMNS]


def reference_calendars(reference_data: pd.DataFrame) -> dict[str, pd.DatetimeIndex]:
    # Build clean-reference session dates for each normalized ticker.
    calendars = {}
    if reference_data.empty:
        return calendars
    valid = reference_data.dropna(subset=["ticker", "ts"])
    for ticker, group in valid.groupby("ticker", sort=True):
        calendars[ticker] = pd.DatetimeIndex(group["ts"].drop_duplicates().sort_values())
    return calendars


def detect_missing_dates(data: pd.DataFrame, calendars: dict[str, pd.DatetimeIndex]) -> pd.DataFrame:
    # Report absent reference sessions inside each ticker's supplied history.
    records = []
    for ticker, group in data.dropna(subset=["ticker", "ts"]).groupby("ticker", sort=True):
        calendar = calendars.get(ticker)
        if calendar is None or calendar.empty:
            continue
        observed = pd.DatetimeIndex(group["ts"].drop_duplicates())
        expected = calendar[(calendar >= observed.min()) & (calendar <= observed.max())]
        for missing_date in expected.difference(observed):
            records.append({
                "ticker": ticker,
                "issue_type": "missing_trading_date",
                "issue_date": missing_date.strftime("%Y-%m-%d"),
                "description": "This ticker-date occurs in the clean ohlc reference within the supplied active history but not in ohlc_hw.",
                "severity": "medium",
            })
    return pd.DataFrame(records, columns=ISSUE_COLUMNS)


def build_quality_report(data: pd.DataFrame, calendars: dict[str, pd.DatetimeIndex], factor_reference: pd.DataFrame) -> pd.DataFrame:
    # Diagnose missing, invalid, inconsistent, duplicate, and unusual observations.
    reports = []
    invalid_ticker = data["ticker"].isna() | data["ticker"].astype("string").str.strip().eq("")
    invalid_date = data["ts"].isna()
    reports.append(issue_rows(data, invalid_ticker, "invalid_ticker", "high", "Ticker is missing after normalization."))
    reports.append(issue_rows(data, invalid_date, "invalid_date", "high", "Date is missing or could not be parsed."))
    for column in PRICE_COLUMNS + ["volume"]:
        missing = data[column].isna()
        reports.append(issue_rows(data, missing, f"missing_{column}", "medium" if column in PRICE_COLUMNS else "low", f"{column} is missing."))
        nonfinite = data[column].notna() & ~np.isfinite(data[column])
        reports.append(issue_rows(data, nonfinite, f"nonfinite_{column}", "high" if column in PRICE_COLUMNS else "medium", f"{column} is non-finite."))
        if column in PRICE_COLUMNS:
            nonpositive = data[column].notna() & data[column].le(0)
            reports.append(issue_rows(data, nonpositive, f"nonpositive_{column}", "high", f"{column} must be positive."))
        else:
            negative = data[column].notna() & data[column].lt(0)
            zero = data[column].notna() & data[column].eq(0)
            reports.append(issue_rows(data, negative, "negative_volume", "medium", "Volume is negative."))
            reports.append(issue_rows(data, zero, "zero_volume", "low", "Zero volume may indicate a stale record or missing-value placeholder."))

    # Check each inequality independently; a missing third field must not hide
    # a valid comparison between two observed prices.
    bounds_checks = [
        ("high_below_low", "high", "low", "lt"),
        ("high_below_open", "high", "open", "lt"),
        ("high_below_close", "high", "close", "lt"),
        ("low_above_open", "low", "open", "gt"),
        ("low_above_close", "low", "close", "gt"),
    ]
    for issue_type, left, right, comparison in bounds_checks:
        observed_pair = np.isfinite(data[left]) & np.isfinite(data[right])
        violates = getattr(data[left], comparison)(data[right])
        reports.append(issue_rows(
            data, observed_pair & violates, issue_type, "high",
            f"{left} {('<' if comparison == 'lt' else '>')} {right}; raw OHLC ordering is impossible.",
        ))

    return_data = data.drop_duplicates(["ticker", "ts", *PRICE_COLUMNS, "volume"], keep="first")
    conflicting = return_data.duplicated(["ticker", "ts"], keep=False)
    valid_keys = return_data.loc[~conflicting].dropna(subset=["ticker", "ts"]).sort_values(["ticker", "ts"]).copy()
    valid_keys["simple_return"] = valid_keys.groupby("ticker", sort=False)["close"].pct_change(fill_method=None)
    valid_returns = valid_keys["simple_return"].replace([np.inf, -np.inf], np.nan)
    means = valid_returns.groupby(valid_keys["ticker"]).transform("mean")
    stds = valid_returns.groupby(valid_keys["ticker"]).transform("std")
    # A ticker with fewer than two defined returns has undefined sample std;
    # a zero std also gives no meaningful threshold. Both cases are explicitly
    # skipped instead of dividing by zero or inventing an outlier flag.
    outlier_mask = valid_returns.notna() & stds.gt(0) & (valid_returns - means).abs().gt(3 * stds)
    reports.append(issue_rows(valid_keys, outlier_mask, "extreme_return", "medium", "Simple close return is more than 3 sample standard deviations from this ticker's mean; review flag only."))

    duplicate_mask = data.duplicated(["ticker", "ts"], keep=False) & data["ticker"].notna() & data["ts"].notna()
    reports.append(issue_rows(data, duplicate_mask, "duplicate_key", "medium", "Repeated normalized ticker/date key; exact duplicates can be removed and conflicting rows require quarantine."))
    reports.append(detect_missing_dates(data, calendars))

    if not factor_reference.empty:
        ref = factor_reference.dropna(subset=["ticker", "ts", "close", "adj_close"]).copy()
        ref = ref.loc[ref["close"].gt(0) & ref["adj_close"].gt(0)].drop_duplicates(["ticker", "ts"])
        ref["reference_factor"] = ref["adj_close"] / ref["close"]
        joined = data[["ticker", "ts", "close", "adj_close"]].merge(ref[["ticker", "ts", "reference_factor"]], on=["ticker", "ts"], how="left")
        expected = joined["close"] * joined["reference_factor"]
        mismatch = joined["reference_factor"].notna() & joined["adj_close"].gt(0) & ~np.isclose(joined["adj_close"], expected, rtol=1e-4, atol=0.02, equal_nan=False)
        mismatch_rows = data.loc[mismatch.to_numpy(), ["ticker", "ts"]].copy()
        if not mismatch_rows.empty:
            mismatch_rows["issue_type"] = "adjusted_close_factor_mismatch"
            mismatch_rows["issue_date"] = mismatch_rows["ts"].dt.strftime("%Y-%m-%d")
            mismatch_rows["description"] = "Vendor adjusted close disagrees with the same-date clean-reference adjustment factor."
            mismatch_rows["severity"] = "medium"
            reports.append(mismatch_rows[ISSUE_COLUMNS])

    return pd.concat(reports, ignore_index=True).sort_values(["ticker", "issue_date", "issue_type"], kind="stable").reset_index(drop=True)


calendars = reference_calendars(reference)
quality_report = build_quality_report(raw, calendars, reference)
print(f"Pre-repair quality report: {len(quality_report):,} issue rows")
display(quality_report.groupby(["issue_type", "severity"], dropna=False).size().rename("count").reset_index())
display(quality_report.head(15))
missing_date_report = quality_report.loc[quality_report["issue_type"].eq("missing_trading_date")]
if missing_date_report.empty:
    print("No missing reference-calendar dates found within supplied ticker histories.")
else:
    print("Missing-date summary:")
    missing_summary = missing_date_report.groupby("ticker").agg(missing_days=("issue_date", "size"), first_missing=("issue_date", "min"), last_missing=("issue_date", "max")).reset_index()
    for row in missing_summary.itertuples(index=False):
        print(f"Ticker {row.ticker} is missing {row.missing_days} trading days between {row.first_missing} and {row.last_missing}")
    display(missing_summary)

Pre-repair quality report: 95,187 issue rows
                       issue_type severity  count
0  adjusted_close_factor_mismatch   medium  11265
1                   duplicate_key   medium  58920
2                  extreme_return   medium   5056
3                  low_above_open     high      5
4               missing_adj_close   medium  19855
5                nonpositive_open     high      5
6                     zero_volume      low     81
   ticker  ... severity
0    AAPL  ...   medium
1    AAPL  ...   medium
2    AAPL  ...   medium
3    AAPL  ...   medium
4    AAPL  ...   medium
5    AAPL  ...   medium
6    AAPL  ...   medium
7    AAPL  ...   medium
8    AAPL  ...   medium
9    AAPL  ...   medium
10   AAPL  ...   medium
11   AAPL  ...   medium
12   AAPL  ...   medium
13   AAPL  ...   medium
14   AAPL  ...   medium

[15 rows x 5 columns]
No missing reference-calendar dates found within supplied ticker histories.


### Small diagnostic example

This fixture deliberately omits one reference session and includes a nonpositive adjusted close, a broken high/low range, and zero volume. It demonstrates issue reporting without introducing synthetic prices into the real dataset.

In [4]:
fixture = normalize_ohlc(pd.DataFrame([
    {"ts": "2024-01-02", "ticker": "ABC", "open": 10, "high": 11, "low": 9, "close": 10, "adj_close": 10, "volume": 0},
    {"ts": "2024-01-04", "ticker": "ABC", "open": 0, "high": 10, "low": 8, "close": 10, "adj_close": -1, "volume": np.nan},
    {"ts": "2024-01-05", "ticker": "ABC", "open": 10, "high": 8, "low": 12, "close": 10, "adj_close": 10, "volume": 100},
]))
fixture_calendar = {"ABC": pd.date_range("2024-01-02", "2024-01-05", freq="D")}
fixture_missing_dates = detect_missing_dates(fixture, fixture_calendar)
assert len(fixture_missing_dates) == 1
assert fixture_missing_dates.iloc[0]["issue_date"] == "2024-01-03"
fixture_report = build_quality_report(fixture, fixture_calendar, pd.DataFrame())
display(fixture_report)
assert fixture_report["issue_type"].eq("missing_trading_date").any()
assert fixture_report["issue_type"].eq("nonpositive_adj_close").any()
for issue_type in [
    "high_below_low", "high_below_open", "high_below_close",
    "low_above_open", "low_above_close",
]:
    assert fixture_report["issue_type"].eq(issue_type).any(), issue_type
assert fixture_report["issue_type"].eq("zero_volume").any()
assert fixture_report["issue_type"].eq("missing_volume").any()

# Run the same ticker worker as the process pool on this tiny injected group.
repair_fixture = fixture.copy()
for flag in REPAIR_FLAG_COLUMNS:
    repair_fixture[flag] = False
repair_fixture["_row_id"] = np.arange(len(repair_fixture))
repair_fixture["_original_close"] = repair_fixture["close"]
repair_fixture["_original_adj_close"] = repair_fixture["adj_close"]
repair_fixture["_reference_factor"] = 1.0
repair_preview, repair_quarantine = repair_ticker_group(repair_fixture)
repaired_day = repair_preview.loc[repair_preview["ts"].astype(str).eq("2024-01-04")].iloc[0]
assert repaired_day["open"] == 9 and repaired_day["open_repaired"]
assert repaired_day["adj_close"] == 10 and repaired_day["adj_close_repaired"]
assert repaired_day["volume"] == 0 and repaired_day["volume_repaired"]
repaired_bounds = repair_preview.loc[repair_preview["ts"].astype(str).eq("2024-01-05")].iloc[0]
assert repaired_bounds["high"] == 10 and repaired_bounds["low"] == 8
assert repaired_bounds["high_repaired"] and repaired_bounds["low_repaired"]
assert not repair_preview["ts"].astype(str).eq("2024-01-03").any()
display(repair_preview[["ticker", "ts", "high", "low", "adj_close", "volume", *REPAIR_FLAG_COLUMNS]])
print("Fixture demonstrates missing-date detection, midpoint repair for an invalid open, and ticker-local forward-fill for impossible bounds; absent dates are not fabricated.")

   ticker  ... severity
0     ABC  ...      low
1     ABC  ...   medium
2     ABC  ...     high
3     ABC  ...      low
4     ABC  ...     high
5     ABC  ...     high
6     ABC  ...     high
7     ABC  ...     high
8     ABC  ...     high
9     ABC  ...     high
10    ABC  ...     high

[11 rows x 5 columns]
  ticker          ts  high  ...  close_repaired  adj_close_repaired  volume_repaired
0    ABC  2024-01-02  11.0  ...           False               False            False
1    ABC  2024-01-04  10.0  ...           False                True             True
2    ABC  2024-01-05  10.0  ...           False               False            False

[3 rows x 12 columns]
Fixture demonstrates missing-date detection, midpoint repair for an invalid open, and ticker-local forward-fill for impossible bounds; absent dates are not fabricated.


## 1.3 Clean the data in parallel

Ticker histories are independent after keys and duplicate records are resolved. The notebook distributes ticker groups to a process pool, using `max(1, os.cpu_count() - 2)` workers. If the platform cannot start that process pool, it falls back to a thread pool with the same worker count. Rows are combined, validated, and written to SQLite in the main process.

Policy: drop rows with all raw OHLC cells missing before filling; remove exact normalized duplicates; quarantine conflicting normalized ticker/date keys. For a nonpositive/non-finite observed open, if positive low, close, and high establish low <= close <= high, set open to (low + close) / 2 and flag it. This midpoint is a documented estimate, not an observed price. Set other invalid raw prices to missing, forward-fill only within a sorted ticker, and do not backfill leading gaps. For an inverted `high < low` range, mark both bounds missing; for `high < open/close` or `low > open/close`, mark the offending bound missing. Carry earlier values only within the same ticker, then re-bound around open/close and flag every changed field. Quarantine any row that remains invalid. Use date-matched reference factors for adjusted-close repairs; do not carry factors between dates. Replace missing or invalid volume with zero only when its repair flag is true. Extreme returns remain review flags unless independent evidence supports a correction.

In [5]:
def trusted_reference_factors(reference_data: pd.DataFrame) -> dict[tuple[str, pd.Timestamp], float]:
    # Return only valid, date-specific factors from the clean reference table.
    if reference_data.empty:
        return {}
    ref = reference_data.dropna(subset=["ticker", "ts", "close", "adj_close"]).copy()
    valid = np.isfinite(ref["close"]) & np.isfinite(ref["adj_close"]) & ref["close"].gt(0) & ref["adj_close"].gt(0)
    ref = ref.loc[valid].drop_duplicates(["ticker", "ts"], keep="first")
    return {(row.ticker, row.ts): row.adj_close / row.close for row in ref.itertuples(index=False)}


def clean_ohlc(source: pd.DataFrame, reference_data: pd.DataFrame):
    # Resolve invalid keys, empty OHLC records, and duplicates before parallel work.
    data = normalize_ohlc(source).reset_index(drop=True)
    for flag in REPAIR_FLAG_COLUMNS:
        data[flag] = False
    data["_row_id"] = np.arange(len(data))
    data["_original_close"] = data["close"]
    data["_original_adj_close"] = data["adj_close"]
    factors = trusted_reference_factors(reference_data)
    data["_reference_factor"] = [factors.get((ticker, ts), np.nan) for ticker, ts in zip(data["ticker"], data["ts"])]
    removed_parts, quarantine_parts = [], []

    invalid_key = data["ticker"].isna() | data["ticker"].str.strip().eq("") | data["ts"].isna()
    if invalid_key.any():
        invalid_rows = data.loc[invalid_key].copy()
        invalid_rows["quarantine_reason"] = "Missing or invalid normalized ticker/date."
        quarantine_parts.append(invalid_rows)
    data = data.loc[~invalid_key].copy()

    all_raw_missing = data[RAW_PRICE_COLUMNS].isna().all(axis=1)
    if all_raw_missing.any():
        dropped = data.loc[all_raw_missing].copy()
        dropped["removal_reason"] = "All raw OHLC fields were missing; removed before filling."
        removed_parts.append(dropped)
    data = data.loc[~all_raw_missing].copy()

    exact_columns = ["ticker", "ts", *PRICE_COLUMNS, "volume"]
    exact_duplicate = data.duplicated(exact_columns, keep="first")
    if exact_duplicate.any():
        dropped = data.loc[exact_duplicate].copy()
        dropped["removal_reason"] = "Exact duplicate after ticker/date normalization."
        removed_parts.append(dropped)
    data = data.loc[~exact_duplicate].copy()

    conflicting = data.duplicated(["ticker", "ts"], keep=False)
    if conflicting.any():
        conflicts = data.loc[conflicting].copy()
        conflicts["quarantine_reason"] = "Conflicting rows share normalized ticker/date; no arbitrary record was chosen."
        quarantine_parts.append(conflicts)
    data = data.loc[~conflicting].sort_values(["ticker", "ts", "_row_id"]).copy()

    ticker_groups = [group.copy() for _, group in data.groupby("ticker", sort=True)]
    workers = max(1, (os.cpu_count() or 1) - 2)
    print(f"Cleaning {len(ticker_groups)} ticker groups with {workers} workers (CPU count minus 2).")
    try:
        process_context = mp.get_context("fork") if "fork" in mp.get_all_start_methods() else None
        with ProcessPoolExecutor(max_workers=workers, mp_context=process_context) as pool:
            results = list(pool.map(repair_ticker_group, ticker_groups))
        print("Ticker repairs completed with a process pool.")
    except (OSError, PermissionError, BrokenProcessPool) as error:
        print(f"Process pool unavailable ({error}); using a thread pool with {workers} workers.")
        with ThreadPoolExecutor(max_workers=workers) as pool:
            results = list(pool.map(repair_ticker_group, ticker_groups))

    cleaned_parts = [result[0] for result in results if not result[0].empty]
    quarantine_parts.extend(result[1] for result in results if not result[1].empty)
    cleaned = pd.concat(cleaned_parts, ignore_index=True) if cleaned_parts else pd.DataFrame()
    cleaned = cleaned.sort_values(["ticker", "ts"]).reset_index(drop=True)
    removed = pd.concat(removed_parts, ignore_index=True) if removed_parts else pd.DataFrame()
    quarantined = pd.concat(quarantine_parts, ignore_index=True) if quarantine_parts else pd.DataFrame()
    counts = {
        "input_rows": len(source),
        "retained_rows": len(cleaned),
        "removed_rows": len(removed),
        "quarantined_rows": len(quarantined),
        "repair_cells": int(cleaned[REPAIR_FLAG_COLUMNS].to_numpy(dtype=bool).sum()),
        "removed_exact_duplicates": int(sum(part["removal_reason"].str.startswith("Exact duplicate").sum() for part in removed_parts if not part.empty)),
        "removed_all_raw_missing": int(sum(part["removal_reason"].str.startswith("All raw").sum() for part in removed_parts if not part.empty)),
    }
    return cleaned.reset_index(drop=True), removed, quarantined, counts


def validate_cleaned(data: pd.DataFrame) -> None:
    # Enforce the cleaned-table invariants before and after persistence.
    if data.empty:
        raise ValueError("No retained rows are available to save.")
    values = data[PRICE_COLUMNS].to_numpy(dtype=float)
    if not np.isfinite(values).all() or not (values > 0).all():
        raise AssertionError("Retained prices must be finite, present, and positive.")
    if data[["ticker", "ts"]].isna().any().any() or data.duplicated(["ticker", "ts"]).any():
        raise AssertionError("Retained ticker/date keys must be complete and unique.")
    if not data["ts"].astype(str).str.fullmatch(r"\d{4}-\d{2}-\d{2}").all():
        raise AssertionError("Dates must use YYYY-MM-DD format.")
    if not np.isfinite(data["volume"].to_numpy(dtype=float)).all() or data["volume"].lt(0).any():
        raise AssertionError("Volume must be finite and nonnegative.")
    bounds = data["low"].le(data["high"]) & data["low"].le(data["open"]) & data["open"].le(data["high"]) & data["low"].le(data["close"]) & data["close"].le(data["high"])
    if not bounds.all():
        raise AssertionError("Retained rows violate OHLC bounds.")


def validate_reference_adjustments(data: pd.DataFrame, reference_data: pd.DataFrame) -> int:
    # Check every cleaned observation with a trusted same-date factor.
    factors = trusted_reference_factors(reference_data)
    dates = pd.to_datetime(data["ts"], errors="coerce").dt.normalize()
    values = np.array([factors.get((ticker, date), np.nan) for ticker, date in zip(data["ticker"], dates)], dtype=float)
    matched = np.isfinite(values) & (values > 0)
    expected = data["close"].to_numpy(dtype=float) * values
    observed = data["adj_close"].to_numpy(dtype=float)
    agrees = np.isclose(observed, expected, rtol=1e-4, atol=0.02, equal_nan=False)
    if np.any(matched & ~agrees):
        raise AssertionError(f"{int((matched & ~agrees).sum())} adjusted closes disagree with date-matched reference factors.")
    return int(matched.sum())


In [6]:
# Inject issues absent from the real data to exercise the full cleaning function.
tiny_source = pd.DataFrame([
    {"ts": "2024-01-02", "ticker": "TST", "open": 10, "high": 11, "low": 9, "close": 10, "adj_close": 10, "volume": 100},
    {"ts": "2024-01-03", "ticker": "TST", "open": np.nan, "high": 11, "low": 9, "close": 10, "adj_close": 10, "volume": np.nan},
    {"ts": "2024-01-04", "ticker": "TST", "open": np.nan, "high": np.nan, "low": np.nan, "close": np.nan, "adj_close": np.nan, "volume": 100},
    {"ts": "2024-01-05", "ticker": "TST", "open": 10, "high": 11, "low": 9, "close": 10, "adj_close": 10, "volume": 100},
    {"ts": "2024-01-05", "ticker": "TST", "open": 10, "high": 11, "low": 9, "close": 10.5, "adj_close": 10.5, "volume": 100},
    {"ts": "2024-01-06", "ticker": "TST", "open": np.inf, "high": 11, "low": 9, "close": 10, "adj_close": 10, "volume": 100},
    {"ts": "2024-01-02", "ticker": "LEAD", "open": np.nan, "high": 11, "low": 9, "close": 10, "adj_close": 10, "volume": 100},
])
tiny_issues = build_quality_report(normalize_ohlc(tiny_source), {}, pd.DataFrame())
print("Injected diagnostic issue types:", sorted(tiny_issues["issue_type"].unique()))
tiny_cleaned, tiny_removed, tiny_quarantine, tiny_counts = clean_ohlc(tiny_source, pd.DataFrame())
assert tiny_counts["input_rows"] == 7
assert tiny_counts["retained_rows"] == 3
assert tiny_counts["removed_rows"] == 1  # all four raw OHLC fields missing
assert tiny_counts["quarantined_rows"] == 3  # two conflicts and one leading gap
assert tiny_cleaned.loc[tiny_cleaned["ts"].eq("2024-01-03"), "open_repaired"].all()
assert tiny_cleaned.loc[tiny_cleaned["ts"].eq("2024-01-03"), "volume_repaired"].all()
assert tiny_cleaned.loc[tiny_cleaned["ts"].eq("2024-01-06"), "open_repaired"].all()
assert set(tiny_quarantine["ticker"]) == {"TST", "LEAD"}
display(pd.Series(tiny_counts, name="rows_or_cells").to_frame())
display(tiny_quarantine[["ticker", "ts", "quarantine_reason"]])
print("Fixture confirms within-ticker fill, missing-volume flag, all-raw-missing removal, conflict quarantine, and leading-gap quarantine.")

Injected diagnostic issue types: ['duplicate_key', 'missing_adj_close', 'missing_close', 'missing_high', 'missing_low', 'missing_open', 'missing_volume', 'nonfinite_open']
Cleaning 2 ticker groups with 10 workers (CPU count minus 2).
Ticker repairs completed with a process pool.
                          rows_or_cells
input_rows                            7
retained_rows                         3
removed_rows                          1
quarantined_rows                      3
repair_cells                          3
removed_exact_duplicates              0
removed_all_raw_missing               1
  ticker         ts                                  quarantine_reason
0    TST 2024-01-05  Conflicting rows share normalized ticker/date;...
1    TST 2024-01-05  Conflicting rows share normalized ticker/date;...
2   LEAD 2024-01-02  A required price remained missing/invalid, or ...
Fixture confirms within-ticker fill, missing-volume flag, all-raw-missing removal, conflict quarantine, and leading-

In [7]:
cleaned, removed_rows, quarantine, cleaning_counts = clean_ohlc(raw_source, reference)
validate_cleaned(cleaned)
reference_rows_checked = validate_reference_adjustments(cleaned, reference)
cleaned_quality_report = build_quality_report(normalize_ohlc(cleaned), calendars, reference)
print(f"Adjusted closes checked against matching reference factors: {reference_rows_checked:,}")
print(f"Post-repair review flags: {len(cleaned_quality_report):,}")
if not cleaned_quality_report.empty:
    display(cleaned_quality_report.groupby(["issue_type", "severity"]).size().rename("count").reset_index())
print("Cleaning counts:")
display(pd.Series(cleaning_counts, name="rows_or_cells").to_frame())
print("Repair flags by field:")
display(cleaned[REPAIR_FLAG_COLUMNS].sum().rename("repaired_rows").to_frame())
if not quarantine.empty:
    print("Quarantine reasons:")
    display(quarantine["quarantine_reason"].value_counts().rename_axis("reason").reset_index(name="rows"))


def price_comparison(before: pd.DataFrame, after: pd.DataFrame) -> pd.DataFrame:
    # Compare row counts and price ranges before and after cleaning.
    rows = []
    for label, frame in [("before", before), ("after", after)]:
        for column in PRICE_COLUMNS:
            values = pd.to_numeric(frame[column], errors="coerce")
            rows.append({"stage": label, "column": column, "rows": len(frame), "missing": int(values.isna().sum()), "min": values.min(), "median": values.median(), "max": values.max()})
    return pd.DataFrame(rows)

before_after = price_comparison(raw, cleaned)
display(before_after)


Cleaning 30 ticker groups with 10 workers (CPU count minus 2).
Ticker repairs completed with a process pool.
Adjusted closes checked against matching reference factors: 368,251
Post-repair review flags: 5,131
       issue_type severity  count
0  extreme_return   medium   5056
1     zero_volume      low     75
Cleaning counts:
                          rows_or_cells
input_rows                       397711
retained_rows                    368251
removed_rows                      29460
quarantined_rows                      0
repair_cells                      28796
removed_exact_duplicates          29460
removed_all_raw_missing               0
Repair flags by field:
                    repaired_rows
open_repaired                   5
high_repaired                   0
low_repaired                    0
close_repaired                  0
adj_close_repaired          28791
volume_repaired                 0
    stage     column    rows  missing       min  median           max
0  before       open 

### Investigate extreme-return review flags

The simple-close return screen is retrospective and cannot identify a corporate action by itself. Compare every flagged ticker/date with the supplied clean `ohlc` reference close. Agreement is evidence against an isolated raw-close transcription error in `ohlc_hw`; it is not proof that the move was caused by news, a split, or a merger. Without event metadata or a confirmed bad price, keep these rows and their review flags rather than interpolating them.

In [8]:
extreme_keys = quality_report.loc[
    quality_report["issue_type"].eq("extreme_return"), ["ticker", "issue_date"]
].drop_duplicates()
reference_close = reference[["ticker", "ts", "close"]].copy()
reference_close["issue_date"] = reference_close["ts"].dt.strftime("%Y-%m-%d")
reference_close = reference_close.rename(columns={"close": "reference_close"})
cleaned_close = cleaned[["ticker", "ts", "close"]].rename(
    columns={"ts": "issue_date", "close": "retained_close"}
)
outlier_audit = extreme_keys.merge(
    cleaned_close, on=["ticker", "issue_date"], how="left"
).merge(
    reference_close[["ticker", "issue_date", "reference_close"]],
    on=["ticker", "issue_date"], how="left"
)
matched_reference = outlier_audit["reference_close"].notna()
close_agrees = np.isclose(
    outlier_audit["retained_close"].to_numpy(dtype=float),
    outlier_audit["reference_close"].to_numpy(dtype=float),
    rtol=1e-10, atol=1e-10, equal_nan=False,
)
print(f"Flagged ticker/dates: {len(outlier_audit):,}")
print(f"Date-matched clean-reference closes: {int(matched_reference.sum()):,}")
print(f"Flagged closes agreeing with that reference: {int((matched_reference & close_agrees).sum()):,}")
if not (matched_reference & close_agrees).all():
    display(outlier_audit.loc[~(matched_reference & close_agrees)].head(10))
else:
    print("No raw-close disagreement was confirmed against the supplied reference; causes remain unverified.")

# Show a few of the largest flagged returns for human review.
retained_for_returns = cleaned.sort_values(["ticker", "ts"]).copy()
retained_for_returns["simple_return"] = retained_for_returns.groupby("ticker")["close"].pct_change(fill_method=None)
review_examples = outlier_audit.merge(
    retained_for_returns[["ticker", "ts", "simple_return"]].rename(columns={"ts": "issue_date"}),
    on=["ticker", "issue_date"], how="left",
)
review_examples["absolute_return"] = review_examples["simple_return"].abs()
display(review_examples.nlargest(10, "absolute_return")[[
    "ticker", "issue_date", "simple_return", "retained_close", "reference_close"
]])

Flagged ticker/dates: 5,056
Date-matched clean-reference closes: 5,056
Flagged closes agreeing with that reference: 5,056
No raw-close disagreement was confirmed against the supplied reference; causes remain unverified.
     ticker  issue_date  simple_return  retained_close  reference_close
4425    TRV  1976-10-28       1.000000        9.220000         9.220000
4429    TRV  1981-08-10       1.000000       12.060000        12.060000
4427    TRV  1978-12-20       0.997512        8.030000         8.030000
3066    MCD  1968-01-02      -0.753620        0.174900         0.174900
98     AAPL  2000-09-29      -0.518695        0.459820         0.459820
4430    TRV  1981-08-11      -0.500000        6.030000         6.030000
4426    TRV  1976-10-29      -0.491323        4.690000         4.690000
4428    TRV  1978-12-21      -0.490660        4.090000         4.090000
3954   NVDA  2000-03-07       0.424106        0.173570         0.173570
3996   NVDA  2004-08-06      -0.352314        0.078584      

## 1.4 Persist and verify

The cleaned table and audit tables are written to this local database. `ohlc_hw` is not updated. The removed-duplicate table preserves rows omitted as exact duplicates, while the quarantine table preserves rows that could not be safely retained.

In [9]:
if len(cleaned) + len(removed_rows) + len(quarantine) != len(raw_source):
    raise AssertionError("Row accounting failed: retained + removed + quarantined must equal input rows.")

with sqlite3.connect(DB_PATH) as connection:
    cleaned.to_sql("ohlc_hw_cleaned", connection, if_exists="replace", index=False)
    quality_report.to_sql("ohlc_hw_quality_report", connection, if_exists="replace", index=False)
    cleaned_quality_report.to_sql("ohlc_hw_cleaned_quality_report", connection, if_exists="replace", index=False)
    (removed_rows if not removed_rows.empty else pd.DataFrame(columns=[*raw_source.columns, "removal_reason"])).to_sql("ohlc_hw_removed_rows", connection, if_exists="replace", index=False)
    (quarantine if not quarantine.empty else pd.DataFrame(columns=[*raw_source.columns, "quarantine_reason"])).to_sql("ohlc_hw_quarantine", connection, if_exists="replace", index=False)

with sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True) as connection:
    persisted = pd.read_sql_query("SELECT * FROM ohlc_hw_cleaned", connection)
    raw_count_after = pd.read_sql_query("SELECT COUNT(*) AS n FROM ohlc_hw", connection).loc[0, "n"]
validate_cleaned(persisted)
reloaded_factor_checks = validate_reference_adjustments(persisted, reference)
assert int(raw_count_after) == len(raw_source), "The original ohlc_hw row count changed."
print(f"Saved and reloaded {len(persisted):,} validated rows from ohlc_hw_cleaned.")
print(f"Raw ohlc_hw remains at {int(raw_count_after):,} rows.")
print(f"Reloaded adjusted-close/reference-factor checks: {reloaded_factor_checks:,}.")
print("Audit tables: ohlc_hw_quality_report, ohlc_hw_cleaned_quality_report, ohlc_hw_removed_rows, ohlc_hw_quarantine.")


Saved and reloaded 368,251 validated rows from ohlc_hw_cleaned.
Raw ohlc_hw remains at 397,711 rows.
Reloaded adjusted-close/reference-factor checks: 368,251.
Audit tables: ohlc_hw_quality_report, ohlc_hw_cleaned_quality_report, ohlc_hw_removed_rows, ohlc_hw_quarantine.


## Summary

The notebook separates absent trading-date rows from missing cells, creates pre-repair issue rows, records field-level repair flags, and accounts for retained, removed, and quarantined records. Price repairs use prior observations only within the same ticker; adjusted-close repairs use same-date factors when available. Extreme-return observations remain review flags rather than automatic deletions.